# fusion-tools Demo

This notebook demonstrates the core functionality of the **fusion-tools** library:

1. **CSV Parser** – load example W7-X signal data and metadata
2. **TIMDR Filter** – smooth and filter the time-series signals
3. **TIMDR Visualizer** – plot raw vs. filtered signals
4. **LATRO Core** – detect and segment anomalous events
5. **LATRO Features** – extract a feature vector from a detected event
6. **Model-J Detector** – run a multi-signal disruption check


In [ ]:
import sys
import os

# Add the repo root to the path so we can import the packages directly
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('fusion-tools demo ready.')

## 1 – Load example data

In [ ]:
from parsers import CSVParser

data_dir = os.path.join(REPO_ROOT, 'data')

parser = CSVParser(
    filepath=os.path.join(data_dir, 'example_w7x_signal.csv'),
    metadata_path=os.path.join(data_dir, 'example_metadata.json'),
)

time     = parser.get_time()
signal_1 = parser.get_signal('signal_1')
metadata = parser.load_metadata()

print(f"Loaded {len(time)} samples from {metadata['device']} shot {metadata['shot_number']}")
print(f"Available signals: {parser.list_signals()}")

## 2 – Apply TIMDR smoothing

In [ ]:
from timdr import TimdrFilter

filt = TimdrFilter(time, signal_1)

smoothed     = filt.moving_average(window_size=5)
exp_smoothed = filt.exponential_smoothing(alpha=0.3)
normalized   = filt.normalize(method='minmax')
baseline_sub = filt.subtract_baseline(baseline_fraction=0.1)

print('Smoothing complete.')
print(f'First 5 raw values:      {signal_1[:5]}')
print(f'First 5 smoothed values: {smoothed[:5]}')

## 3 – Visualize raw vs. filtered

In [ ]:
import matplotlib
matplotlib.use('Agg')   # use non-interactive backend for demo
import matplotlib.pyplot as plt

from timdr import TimdrVisualizer

viz = TimdrVisualizer(time, signal_1, label='Te', units='keV')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

viz.plot_signal(ax=axes[0])
viz.plot_comparison(smoothed, filtered_label='Moving Avg (w=5)', ax=axes[1])

plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'demo', 'example_plots', 'signal_comparison.png'),
            dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved.')

## 4 – Anomaly detection with LATRO Core

In [ ]:
from latro import LatroCore

core = LatroCore(time, signal_1, threshold=1.5)

anomaly_indices = core.detect_anomalies()
events          = core.segment_events(min_gap=3)
summary         = core.event_summary()

print(f"Anomalous samples : {summary['n_anomalous_samples']}")
print(f"Detected events   : {summary['n_events']}")
print(f"Peak anomaly time : {summary['peak_time']:.4f} s")
print(f"Peak anomaly score: {summary['peak_score']:.4f}")

## 5 – Feature extraction with LATRO Features

In [ ]:
from latro import LatroFeatures

features = LatroFeatures(time, signal_1)
fvec = features.feature_vector()

print('Feature vector:')
for k, v in fvec.items():
    print(f'  {k:<22} {v:.6g}')

## 6 – Disruption warning with Model-J

In [ ]:
from model_j import ModelJDetector

detector = ModelJDetector(
    thresholds={
        'signal_1': (0.01, 0.15),
        'signal_2': (-0.01, 0.10),
    },
    warning_window=3,
)

detector.set_time(time)
detector.add_signal('signal_1', signal_1)
detector.add_signal('signal_2', parser.get_signal('signal_2'))

model_summary = detector.summary()

print('Model-J summary:')
for k, v in model_summary.items():
    print(f'  {k}: {v}')